# 05 — Counterfactual Explanations

Counterfactuals answer the question: *what is the smallest change to this input that would change the model's decision?*

## DiCE, NICE, and Actionability Constraints

Counterfactual explanations (Wachter et al., 2017) provide **recourse** — they tell a user not just why they were denied, but what to change. Unlike feature importance scores, counterfactuals are action-oriented.

**DiCE** (Diverse Counterfactual Explanations, Mothilal et al., 2020) generates multiple diverse counterfactuals using an optimisation objective that balances:
- Proximity: counterfactual should be close to the query point
- Diversity: counterfactuals should be spread in feature space (covering different recourse paths)
- Actionability: some features (e.g., age, race) are immutable — these get zero gradient

**NICE** (Nearest Instance Counterfactual Explanations) is a search-based alternative: it finds the nearest training example with the desired class label, then minimises feature changes.

The actionability constraint is critical for regulatory compliance. GDPR Article 22 requires that automated decisions be explainable and that subjects can contest them — counterfactuals provide the actionable path.

In [ ]:
# DiCE-style counterfactual generation
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)

# Simulate a loan approval model
# Features: income, credit_score, debt_ratio, years_employed, loan_amount
feature_names = ['income', 'credit_score', 'debt_ratio', 'years_employed', 'loan_amount']
immutable_features = []  # none immutable in this demo; age would be
actionable_mask = torch.ones(5)  # 1=actionable

X, y = make_classification(n_samples=1000, n_features=5, n_informative=4,
                            n_redundant=1, random_state=42)
scaler = StandardScaler()
X = scaler.fit_transform(X)

Xtr = torch.tensor(X[:800], dtype=torch.float32)
ytr = torch.tensor(y[:800], dtype=torch.float32)
Xte = torch.tensor(X[800:], dtype=torch.float32)

class LoanModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

model = LoanModel()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for _ in range(100):
    model.train()
    loss = nn.BCELoss()(model(Xtr), ytr)
    opt.zero_grad(); loss.backward(); opt.step()

model.eval()
with torch.no_grad():
    acc = ((model(Xte) > 0.5).float() == torch.tensor(y[800:], dtype=torch.float32)).float().mean()
print(f'Loan model accuracy: {acc:.3f}')

In [ ]:
# Counterfactual search via gradient descent (DiCE-style)
def generate_counterfactuals(model, x_query, target_class=1, n_cf=3,
                              actionable_mask=None, n_steps=500, lr=0.05,
                              proximity_weight=1.0, diversity_weight=0.5):
    """
    Generate diverse counterfactuals via joint optimisation.
    """
    if actionable_mask is None:
        actionable_mask = torch.ones(x_query.shape[0])

    # Initialise counterfactuals near the query
    cfs = x_query.unsqueeze(0).expand(n_cf, -1).clone()
    cfs = cfs + torch.randn_like(cfs) * 0.1
    cfs.requires_grad_(True)

    opt_cf = torch.optim.Adam([cfs], lr=lr)

    for step in range(n_steps):
        pred = model(cfs)
        # Prediction loss: push toward target class
        target = torch.full((n_cf,), float(target_class))
        pred_loss = nn.BCELoss()(pred, target)

        # Proximity loss: stay close to query
        diff = (cfs - x_query) * actionable_mask
        proximity_loss = (diff ** 2).mean()

        # Diversity loss: push CFs apart from each other
        diversity_loss = torch.tensor(0.0)
        for i in range(n_cf):
            for j in range(i+1, n_cf):
                dist = ((cfs[i] - cfs[j]) ** 2).sum() + 1e-6
                diversity_loss += 1.0 / dist

        loss = pred_loss + proximity_weight * proximity_loss + diversity_weight * diversity_loss
        opt_cf.zero_grad(); loss.backward(); opt_cf.step()

        # Freeze immutable features
        with torch.no_grad():
            immutable_idx = (actionable_mask == 0).nonzero(as_tuple=True)[0]
            cfs[:, immutable_idx] = x_query[immutable_idx]

    return cfs.detach()

# Find a denied applicant and generate counterfactuals
with torch.no_grad():
    preds = model(Xte)
denied_idx = (preds < 0.5).nonzero(as_tuple=True)[0][0].item()
x_denied = Xte[denied_idx]
print(f'Denied applicant score: {preds[denied_idx].item():.3f}')

cfs = generate_counterfactuals(model, x_denied, target_class=1, n_cf=3)

print('\nCounterfactual explanations (what to change for approval):')
print(f'{"Feature":<16} {"Original":>10} ' + ' '.join([f'{"CF"+str(i+1):>10}' for i in range(3)]))
for j, fname in enumerate(feature_names):
    orig = x_denied[j].item()
    cf_vals = ' '.join([f'{cfs[i,j].item():>10.3f}' for i in range(3)])
    print(f'{fname:<16} {orig:>10.3f} {cf_vals}')

with torch.no_grad():
    cf_scores = model(cfs)
print('\nCF approval probabilities:', [f'{s:.3f}' for s in cf_scores.numpy()])

In [ ]:
# Visualise feature changes needed
import matplotlib.pyplot as plt

diffs = (cfs - x_denied).numpy()
fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(5)
width = 0.25
for i in range(3):
    ax.bar(x_pos + i*width, diffs[i], width, label=f'CF {i+1}', alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x_pos + width)
ax.set_xticklabels(feature_names)
ax.set_ylabel('Feature change required')
ax.set_title('Counterfactual Recourse Paths — Denied Loan Application')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/counterfactuals.png', dpi=100, bbox_inches='tight')
plt.show()
print('Counterfactual chart saved')